In [8]:
import torch, torchvision
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

torch: 2.6.0+cu126
torchvision: 0.21.0+cu126
CUDA: True
GPU: NVIDIA H100 80GB HBM3


In [9]:
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset

from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    T5ForConditionalGeneration,
    T5TokenizerFast,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

PROJECT_ROOT = Path('.').resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR = PROJECT_ROOT / 'checkpoints' / 'run2'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = PROCESSED_DIR / 'wikisql_t5_train.pt'
VAL_FILE = PROCESSED_DIR / 'wikisql_t5_val.pt'

#baseline model config
MODEL_NAME = 't5-base'
LR = 3e-4
BATCH_SIZE = 32
EPOCHS = 10
WARMUP_STEPS = 500
MAX_GEN_LEN = 128

print('Project root:', PROJECT_ROOT)
print('Train file:', TRAIN_FILE)
print('Val file:', VAL_FILE)
print('Output dir:', OUTPUT_DIR)

Project root: /projects/e32706/sct1077
Train file: /projects/e32706/sct1077/data/processed/wikisql_t5_train.pt
Val file: /projects/e32706/sct1077/data/processed/wikisql_t5_val.pt
Output dir: /projects/e32706/sct1077/checkpoints/run2


In [10]:
class TensorDictDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
        self.length = encodings['input_ids'].size(0)

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.encodings['labels'][idx],
        }


def load_tensor_dict(path: Path):
    if not path.exists():
        raise FileNotFoundError(f'Missing preprocessed file: {path}')
    data = torch.load(path, weights_only=False)
    required = {'input_ids', 'attention_mask', 'labels'}
    if not required.issubset(set(data.keys())):
        raise ValueError(f'{path} missing keys. Expected: {required}')
    return data


train_enc = load_tensor_dict(TRAIN_FILE)
val_enc = load_tensor_dict(VAL_FILE)

train_dataset = TensorDictDataset(train_enc)
val_dataset = TensorDictDataset(val_enc)

print('Train examples:', len(train_dataset))
print('Validation examples:', len(val_dataset))

Train examples: 56355
Validation examples: 8421


In [11]:
tokenizer = T5TokenizerFast.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=32,
    lora_alpha=64,
    target_modules=['q', 'k', 'v', 'o', 'wi', 'wo'],  
    lora_dropout=0.1,
)

model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights: 100%|██████████| 257/257 [00:00<00:00, 12279.97it/s]


trainable params: 12,976,128 || all params: 235,879,680 || trainable%: 5.5012


In [12]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    if predictions.ndim == 3:
        predictions = np.argmax(predictions, axis=-1)

    pad_id = tokenizer.pad_token_id
    vocab_size = tokenizer.vocab_size

    predictions = np.asarray(predictions, dtype=np.int64)
    labels = np.asarray(labels, dtype=np.int64)

    predictions = np.where(predictions == -100, pad_id, predictions)
    labels = np.where(labels == -100, pad_id, labels)

    predictions = np.clip(predictions, 0, vocab_size - 1)
    labels = np.clip(labels, 0, vocab_size - 1)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    exact_match = sum(
        pred.strip() == label.strip()
        for pred, label in zip(decoded_preds, decoded_labels)
    ) / max(1, len(decoded_preds))

    return {'exact_match': exact_match}

In [13]:
use_bf16 = torch.cuda.is_available()

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    warmup_steps=WARMUP_STEPS,
    learning_rate=LR,
    fp16=False,
    bf16=use_bf16,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='exact_match',
    greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=MAX_GEN_LEN,
    logging_steps=100,
    save_total_limit=2,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print('Trainer initialized. Ready to train.')

[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Trainer initialized. Ready to train.


In [14]:
train_result = trainer.train()

print('\nTraining complete.')
print('Train metrics:', train_result.metrics)

eval_metrics = trainer.evaluate(max_length=MAX_GEN_LEN)
print('Validation metrics:', eval_metrics)

trainer.save_model()
tokenizer.save_pretrained(OUTPUT_DIR)
print('Saved model artifacts to', OUTPUT_DIR)

Epoch,Training Loss,Validation Loss,Exact Match
1,0.144008,0.099232,0.468590
2,0.099113,0.074008,0.552785
3,0.083592,0.060953,0.615485
4,0.071635,0.057965,0.628548
5,0.063280,0.056253,0.640423
6,0.059612,0.053958,0.652060
7,0.057812,0.052518,0.656098
8,0.053732,0.051732,0.658829
9,0.052637,0.051018,0.670704
10,0.049119,0.050640,0.670229



Training complete.
Train metrics: {'train_runtime': 3545.2152, 'train_samples_per_second': 158.961, 'train_steps_per_second': 4.97, 'total_flos': 1.828214121037824e+17, 'train_loss': 0.10667843037918126, 'epoch': 10.0}


Training Loss,Validation Loss,Epoch,Exact Match
0.049119,0.051018,10,0.670704


Validation metrics: {'eval_loss': 0.05101848393678665, 'eval_exact_match': 0.6707041919011993}
Saved model artifacts to /projects/e32706/sct1077/checkpoints/run2


In [15]:
import matplotlib.pyplot as plt

history = trainer.state.log_history

train_steps = [entry['step'] for entry in history if 'loss' in entry]
train_losses = [entry['loss'] for entry in history if 'loss' in entry]

eval_epochs = [entry['epoch'] for entry in history if 'eval_loss' in entry]
eval_losses = [entry['eval_loss'] for entry in history if 'eval_loss' in entry]
eval_exact_match = [entry.get('eval_exact_match') for entry in history if 'eval_loss' in entry]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_steps, train_losses, color='tab:blue')
axes[0].set_xlabel('Training step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)

if eval_epochs:
    axes[1].plot(eval_epochs, eval_losses, 'o-', color='tab:orange', label='Val loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss', color='tab:orange')
    axes[1].tick_params(axis='y', labelcolor='tab:orange')
    axes[1].grid(True, alpha=0.3)

    if any(m is not None for m in eval_exact_match):
        ax2 = axes[1].twinx()
        ax2.plot(eval_epochs, eval_exact_match, 's--', color='tab:green', label='Exact match')
        ax2.set_ylabel('Exact Match', color='tab:green')
        ax2.tick_params(axis='y', labelcolor='tab:green')

    axes[1].set_title('Validation Loss & Exact Match')
else:
    axes[1].set_visible(False)

plt.tight_layout()
plt.show()

print(f'Final train loss: {train_losses[-1]:.4f}')
if eval_losses:
    print(f'Final val loss: {eval_losses[-1]:.4f}')
if eval_exact_match and eval_exact_match[-1] is not None:
    print(f'Final val exact match: {eval_exact_match[-1]:.4f}')

ModuleNotFoundError: No module named 'matplotlib'